In [3]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Paths set successfully.")

Paths set successfully.


In [5]:
city_df = pd.read_excel(RAW_PATH + "City.xlsx")
continent_df = pd.read_excel(RAW_PATH + "Continent.xlsx")
country_df = pd.read_excel(RAW_PATH + "Country.xlsx")
item_df = pd.read_excel(RAW_PATH + "Updated_Item.xlsx")
mode_df = pd.read_excel(RAW_PATH + "Mode.xlsx")
region_df = pd.read_excel(RAW_PATH + "Region.xlsx")
transaction_df = pd.read_excel(RAW_PATH + "Transaction.xlsx")
type_df = pd.read_excel(RAW_PATH + "Type.xlsx")
user_df = pd.read_excel(RAW_PATH + "User.xlsx")

print("All files loaded successfully.")

All files loaded successfully.


In [6]:
dfs = [
    city_df, continent_df, country_df, item_df,
    mode_df, region_df, transaction_df,
    type_df, user_df
]

for df in dfs:
    df.drop_duplicates(inplace=True)

print("Duplicates removed.")

Duplicates removed.


In [7]:
for df in dfs:
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna("Unknown")

print("Missing values handled.")

Missing values handled.


In [8]:
print("Transaction Columns:", transaction_df.columns.tolist())
print("User Columns:", user_df.columns.tolist())
print("Item Columns:", item_df.columns.tolist())
print("Type Columns:", type_df.columns.tolist())
print("Mode Columns:", mode_df.columns.tolist())
print("City Columns:", city_df.columns.tolist())
print("Country Columns:", country_df.columns.tolist())
print("Region Columns:", region_df.columns.tolist())
print("Continent Columns:", continent_df.columns.tolist())

Transaction Columns: ['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitMode', 'AttractionId', 'Rating']
User Columns: ['UserId', 'ContinentId', 'RegionId', 'CountryId', 'CityId']
Item Columns: ['AttractionId', 'AttractionCityId', 'AttractionTypeId', 'Attraction', 'AttractionAddress']
Type Columns: ['AttractionTypeId', 'AttractionType']
Mode Columns: ['VisitModeId', 'VisitMode']
City Columns: ['CityId', 'CityName', 'CountryId']
Country Columns: ['CountryId', 'Country', 'RegionId']
Region Columns: ['Region', 'RegionId', 'ContinentId']
Continent Columns: ['ContinentId', 'Continent']


In [9]:
# Start with transaction table
final_df = transaction_df.copy()

# Merge user data
final_df = final_df.merge(user_df, on="UserId", how="left")

# Merge item data
final_df = final_df.merge(item_df, on="AttractionId", how="left")

# Merge attraction type data
final_df = final_df.merge(type_df, on="AttractionTypeId", how="left")

# Merge visit mode data
# FIX: Merge on VisitModeId instead of VisitMode
if "VisitModeId" in final_df.columns and "VisitModeId" in mode_df.columns:
    final_df = final_df.merge(
        mode_df,
        on="VisitModeId",
        how="left",
        suffixes=("", "_mode")
    )

# Merge city data
if "AttractionCityId" in final_df.columns and "CityId" in city_df.columns:
    final_df = final_df.merge(
        city_df,
        left_on="AttractionCityId",
        right_on="CityId",
        how="left",
        suffixes=("", "_city")
    )

# Merge country data
if "CountryId" in final_df.columns and "CountryId" in country_df.columns:
    final_df = final_df.merge(
        country_df,
        on="CountryId",
        how="left",
        suffixes=("", "_country")
    )

# Merge region data
if "RegionId" in final_df.columns and "RegionId" in region_df.columns:
    final_df = final_df.merge(
        region_df,
        on="RegionId",
        how="left",
        suffixes=("", "_region")
    )

# Merge continent data
if "ContinentId" in final_df.columns and "ContinentId" in continent_df.columns:
    final_df = final_df.merge(
        continent_df,
        on="ContinentId",
        how="left",
        suffixes=("", "_continent")
    )

print("final_df created successfully!")
print("Shape:", final_df.shape)
display(final_df.head())

final_df created successfully!
Shape: (52930, 24)


,TransactionId,UserId,VisitYear,VisitMonth,VisitMode,AttractionId,Rating,ContinentId,RegionId,CountryId,CityId,AttractionCityId,AttractionTypeId,Attraction,AttractionAddress,AttractionType,CityId_city,CityName,CountryId_city,Country,RegionId_country,Region,ContinentId_region,Continent
0,3,70456,2022,10,2,640,5,5,21,163,4341.0,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,United Kingdom,21,Western Europe,5,Europe
1,8,7567,2022,10,4,640,5,2,8,48,464.0,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Canada,8,Northern America,2,America
2,9,79069,2022,10,3,640,5,2,9,54,774.0,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Brazil,9,South America,2,America
3,10,31019,2022,10,3,640,3,5,17,135,583.0,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,Switzerland,17,Central Europe,5,Europe
4,15,43611,2022,10,2,640,3,5,21,163,1396.0,1,63,Sacred Monkey Forest Sanctuary,"Jl. Monkey Forest, Ubud 80571 Indonesia",Nature & Wildlife Areas,1,Douala,1,United Kingdom,21,Western Europe,5,Europe


In [8]:
print(type(final_df))
print(final_df.shape)

<class 'pandas.DataFrame'>
(52930, 25)


In [9]:
output_path = "../data/processed/final_processed_data.csv"

final_df.to_csv(output_path, index=False)

print("Processed dataset saved successfully!")
print("Saved to:", output_path)
print("Shape:", final_df.shape)

Processed dataset saved successfully!
Saved to: ../data/processed/final_processed_data.csv
Shape: (52930, 25)


In [10]:
print(os.listdir("../data/processed"))

['final_processed_data.csv']
